# OCR engine comparison

Run one OCR engine against the labeled pages attached to this private Kaggle kernel. Each bundle contains one language-specific ground-truth JSON and its source PDF. Interpret results using the dataset's reference quality and limitations; a small page sample is exploratory and cannot select a production engine.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

input_root = Path("/kaggle/input")
bundle_files = sorted(input_root.rglob("project.zip")) + sorted(input_root.rglob("benchmark_project.zip"))
for bundle in bundle_files:
    extracted = Path("/kaggle/working/benchmark-assets/project")
    shutil.rmtree(extracted.parent, ignore_errors=True)
    shutil.unpack_archive(str(bundle), str(extracted.parent))
project_candidates = {
    *input_root.glob("*/project"),
    *input_root.glob("datasets/*/*/project"),
    *Path("/kaggle/working/benchmark-assets").glob("project"),
    *(entry.parent.parent.parent for entry in input_root.rglob("evaluate_ocr.py") if entry.parent.name == "processing" and entry.parent.parent.name == "scripts"),
}
assets_candidates = sorted(project for project in project_candidates if (project / "scripts/processing/evaluate_ocr.py").is_file())
if len(assets_candidates) != 1:
    raise FileNotFoundError(f"Expected one attached Commerce OCR dataset; found: {assets_candidates}")
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent


In [ ]:
ENGINE = "tesseract"  # Run one engine per kernel: tesseract, paddleocr, easyocr, or surya.
REQUIRE_GPU = False  # Set True only for EasyOCR/Surya GPU runs.
DPI = 180
TESSERACT_PSM = 3
label_files = sorted(Path("data/evaluation/ocr").glob("*_ground_truth.json"))
label_files += sorted(Path("data/evaluation/ocr").glob("*_transcriptions.json"))
if len(label_files) != 1:
    raise FileNotFoundError(f"Attach a benchmark bundle with exactly one OCR labels JSON; found: {label_files}")
DATASET_PATH = label_files[0].as_posix()
DATASET_TAG = label_files[0].stem.removesuffix("_ground_truth").removesuffix("_transcriptions")


In [ ]:
import subprocess, sys

# Install one engine only, avoiding incompatible OCR framework combinations.
subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "--default-timeout=300", "--retries=5", "install", "paddlepaddle==3.3.0"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1", "transformers==4.57.3"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")


In [ ]:
import os

os.environ["HF_HOME"] = "/kaggle/working/model-cache/huggingface"
os.environ["TORCH_HOME"] = "/kaggle/working/model-cache/torch"
if ENGINE == "surya":
    # Fit Surya's detection/recognition batches within Kaggle's 15 GB Tesla T4.
    os.environ["DETECTOR_BATCH_SIZE"] = "16"
    os.environ["RECOGNITION_BATCH_SIZE"] = "128"
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
if ENGINE in {"easyocr", "surya"}:
    import torch
    cuda_available = torch.cuda.is_available()
    print("torch", torch.__version__, "CUDA", cuda_available)
    if REQUIRE_GPU and not cuda_available:
        raise RuntimeError("GPU requested for this OCR engine but CUDA is unavailable; restart with a Kaggle GPU accelerator.")
elif ENGINE == "paddleocr":
    import paddle
    print("paddle", paddle.__version__, "CUDA build", paddle.is_compiled_with_cuda(), "device", paddle.device.get_device())


In [ ]:
import json
dataset_path = Path(DATASET_PATH)
if not dataset_path.is_file():
    raise FileNotFoundError(f"Missing OCR labels: {dataset_path}")
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
for source in sorted({page["file"] for page in dataset["pages"]}):
    if not Path(source).is_file():
        raise FileNotFoundError(f"Missing benchmark source PDF: {source}")
output = Path("/kaggle/working") / f"commerce-ocr-{DATASET_TAG}-{ENGINE}.json"
command = [sys.executable, "scripts/processing/evaluate_ocr.py", "--dataset", DATASET_PATH, "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)]
if ENGINE == "tesseract":
    command.extend(["--tesseract-psm", str(TESSERACT_PSM)])
subprocess.run(command, check=True)
print(f"Download the result from Kaggle Outputs: {output}")


The evaluator selects OCR language from the labeled pages and reports text CER/WER plus ordered table-row and table-cell text recall when table anchors exist. Scores assess OCR text recognition only; they do not evaluate layout detection, column/value association, or the complete upload pipeline. Check the attached dataset's `reference_quality`, `primary_metric_selection_eligible`, and `limitations` before interpreting results. A single-reviewer or synthetic fixture is exploratory and cannot select a production engine. Use the separate upload-path test for parser, table extraction, and persistence.